### Initial ML Model for Delsys Data Input (LDA Model)
Consists of:
- Filtering
- Windowing
- Feature Extraction
- Training LDA
- Evaluating accuracy
- Works for ADLs like “water-bottle lift” and “zipper”

First setting up virtual environment:
- py -m venv venv
- venv\Scripts\activate

Then setting up dependencies:
- pip install numpy scipy scikit-learn matplotlib seaborn

##### Imports

In [1]:
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt, iirnotch
from scipy.stats import skew, kurtosis
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                            accuracy_score, f1_score, precision_recall_fscore_support)
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

##### Functions

In [2]:
def load_all_emg_sensors(csv_path, max_sensors=None):
    """
    Load ALL available EMG sensors from a Delsys Trigno CSV file.
    
    Args:
        csv_path: Path to the CSV file
        max_sensors: Optional limit on number of sensors (None = use all)
    
    Returns:
        emg_data: (n_samples, n_channels) array
        time_data: (n_samples, n_channels) array or None
        sensor_info: dict with metadata about loaded sensors
    """
    # Read all lines to handle variable column counts
    with open(csv_path, 'r') as f:
        lines = f.readlines()
    
    # Find the header line (contains "EMG" and "Time Series")
    header_idx = None
    for i, line in enumerate(lines):
        if "EMG" in line and "Time Series" in line:
            header_idx = i
            break
    
    if header_idx is None:
        raise ValueError("Could not find header row with EMG columns")
    
    # Read from the header row onwards
    df = pd.read_csv(
        csv_path, 
        skiprows=header_idx,
        header=0,
        on_bad_lines='skip',
        low_memory=False
    )
    
    # Clean column names - remove leading/trailing spaces and extra whitespace
    df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)
    
    # Find the first row with actual data
    data_start = None
    for i in range(min(5, len(df))):
        try:
            float(df.iloc[i, 0])
            data_start = i
            break
        except (ValueError, TypeError):
            continue
    
    if data_start is None:
        data_start = 2
    
    df = df.iloc[data_start:].reset_index(drop=True)
    
    # Find ALL EMG columns - look for columns with "(mV)" in the name
    emg_cols = [col for col in df.columns if "(mV)" in col]
    
    # Find corresponding time series columns
    time_cols = [col for col in df.columns if "EMG" in col and "Time Series" in col]
    
    if len(emg_cols) == 0:
        raise ValueError("No EMG columns found. Check CSV format.")
    
    # Optionally limit number of sensors
    if max_sensors is not None:
        emg_cols = emg_cols[:max_sensors]
        time_cols = time_cols[:max_sensors] if len(time_cols) >= max_sensors else time_cols
    
    # Convert to numeric and handle NaN
    for col in emg_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Extract data and remove rows with NaN
    emg_data = df[emg_cols].dropna().to_numpy()
    
    # Extract time data if available
    if time_cols:
        for col in time_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        time_data = df[time_cols].dropna().to_numpy()
    else:
        time_data = None
    
    # Store sensor information
    sensor_info = {
        'n_sensors': len(emg_cols),
        'sensor_names': emg_cols,
        'n_samples': emg_data.shape[0],
        'data_range': (np.min(emg_data), np.max(emg_data))
    }
    
    print(f"✓ Loaded {len(emg_cols)} EMG channels with {emg_data.shape[0]} samples")
    print(f"  Data range: [{sensor_info['data_range'][0]:.4f}, {sensor_info['data_range'][1]:.4f}] mV")
    
    return emg_data, time_data, sensor_info

In [3]:
def bandpass_filter(data, lowcut=20, highcut=450, fs=963, order=4):
    """
    Apply bandpass filter to remove noise and keep EMG signal range.
    EMG signals typically contain useful information between 20-450 Hz.
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data, axis=0)

In [4]:
def notch_filter(data, freq=60, fs=963, Q=30):
    """
    Apply notch filter to remove powerline interference.
    """
    b, a = iirnotch(freq, Q, fs)
    return filtfilt(b, a, data, axis=0)

In [5]:
def rectify_signal(data):
    """
    Full-wave rectification: convert signal to absolute values.
    """
    return np.abs(data)

In [6]:
def extract_enhanced_features(emg_window, fs=963):
    """
    Extract comprehensive EMG features from a window.
    
    Time-domain features (per channel):
        - MAV: Mean Absolute Value
        - RMS: Root Mean Square
        - WL: Waveform Length
        - VAR: Variance
        - STD: Standard Deviation
        - ZC: Zero Crossings (normalized)
        - SSC: Slope Sign Changes (normalized)
        - IEMG: Integrated EMG
        - SKEW: Skewness
        - KURT: Kurtosis
        - WAMP: Willison Amplitude (threshold-based activity)
        
    Frequency-domain features (per channel):
        - MNF: Mean Frequency
        - MDF: Median Frequency
        - PKF: Peak Frequency
        
    Args:
        emg_window: (n_samples, n_channels) array
        fs: sampling frequency
    
    Returns:
        feature_vector: flattened array of all features
    """
    n_channels = emg_window.shape[1]
    
    # Time-domain features
    mav = np.mean(np.abs(emg_window), axis=0)
    rms = np.sqrt(np.mean(emg_window**2, axis=0))
    wl = np.sum(np.abs(np.diff(emg_window, axis=0)), axis=0)
    var = np.var(emg_window, axis=0)
    std = np.std(emg_window, axis=0)
    
    # Zero crossings (with threshold to avoid noise)
    threshold = 0.01
    zc = np.sum(np.abs(np.diff(np.sign(emg_window), axis=0)) > 0, axis=0) / emg_window.shape[0]
    
    # Slope sign changes
    diff_signal = np.diff(emg_window, axis=0)
    ssc = np.sum(np.abs(np.diff(np.sign(diff_signal), axis=0)) > 0, axis=0) / (emg_window.shape[0] - 1)
    
    # Integrated EMG
    iemg = np.sum(np.abs(emg_window), axis=0)
    
    # Statistical features
    skewness = np.array([skew(emg_window[:, i]) for i in range(n_channels)])
    kurt = np.array([kurtosis(emg_window[:, i]) for i in range(n_channels)])
    
    # Willison Amplitude (counts threshold crossings)
    wamp_threshold = 0.02
    wamp = np.sum(np.abs(np.diff(emg_window, axis=0)) > wamp_threshold, axis=0) / emg_window.shape[0]
    
    # Frequency-domain features
    fft_vals = np.fft.rfft(emg_window, axis=0)
    power_spectrum = np.abs(fft_vals)**2
    freqs = np.fft.rfftfreq(emg_window.shape[0], 1/fs)
    
    # Mean frequency
    mnf = np.sum(freqs[:, np.newaxis] * power_spectrum, axis=0) / (np.sum(power_spectrum, axis=0) + 1e-10)
    
    # Median frequency
    cumsum_power = np.cumsum(power_spectrum, axis=0)
    total_power = cumsum_power[-1, :]
    mdf = np.array([freqs[np.argmax(cumsum_power[:, i] >= total_power[i]/2)] 
                    for i in range(n_channels)])
    
    # Peak frequency
    pkf = freqs[np.argmax(power_spectrum, axis=0)]
    
    # Combine all features
    features = np.concatenate([
        mav, rms, wl, var, std, zc, ssc, iemg, 
        skewness, kurt, wamp, mnf, mdf, pkf
    ])
    
    return features

In [7]:
def window_emg(emg_data, fs=963, window_sec=0.25, overlap_sec=0.15, 
               feature_func=extract_enhanced_features):
    """
    Slice EMG data into overlapping windows and extract features.
    
    Returns:
        features: (n_windows, n_features)
        window_indices: (n_windows, 2) start and end indices of each window
    """
    win_size = int(window_sec * fs)
    step = int((window_sec - overlap_sec) * fs)
    features = []
    window_indices = []

    for start in range(0, emg_data.shape[0] - win_size, step):
        window = emg_data[start:start + win_size]
        feat = feature_func(window, fs=fs)
        features.append(feat)
        window_indices.append([start, start + win_size])
    
    return np.array(features), np.array(window_indices)

In [8]:
def process_trial(csv_path, label, fs=963, apply_filters=True, max_sensors=None):
    """
    Load EMG, apply filtering, windowing, and return features + labels.
    Now uses ALL available sensors.
    """
    emg_data, _, sensor_info = load_all_emg_sensors(csv_path, max_sensors=max_sensors)
    
    if apply_filters:
        # Apply preprocessing pipeline
        emg_data = bandpass_filter(emg_data, fs=fs)
        emg_data = notch_filter(emg_data, fs=fs)
        emg_data = rectify_signal(emg_data)
    
    X, window_idx = window_emg(emg_data, fs=fs)
    y = np.full(X.shape[0], label)
    
    return X, y, sensor_info

In [9]:
def load_all_trials(trial_files, labels, max_sensors=None):
    """
    Load multiple trials with their labels.
    
    Args:
        trial_files: list of CSV paths
        labels: list of integer labels for each trial
        max_sensors: Optional limit on number of sensors
    
    Returns:
        X: feature matrix
        y: label vector
        trial_info: dict with information about each trial
    """
    X_list, y_list = [], []
    trial_info = {}
    
    for i, (file, label) in enumerate(zip(trial_files, labels)):
        print(f"\nProcessing trial {i+1}/{len(trial_files)}: {file}")
        X_trial, y_trial, sensor_info = process_trial(file, label, max_sensors=max_sensors)
        X_list.append(X_trial)
        y_list.append(y_trial)
        trial_info[file] = {
            'label': label,
            'n_windows': X_trial.shape[0],
            'sensor_info': sensor_info
        }
    
    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    
    print(f"\n{'='*60}")
    print(f"Total dataset: {X.shape[0]} windows, {X.shape[1]} features")
    print(f"{'='*60}")
    
    return X, y, trial_info

In [10]:
def prepare_data(X, y, test_size=0.2, random_state=42, scaler_type='standard', 
                balance_classes=False):
    """
    Split and scale data with optional class balancing.
    
    Args:
        X: feature matrix
        y: labels
        test_size: proportion of test set
        random_state: random seed
        scaler_type: 'standard' (z-score), 'robust', or 'minmax'
        balance_classes: whether to apply class weights for imbalanced data
    
    Returns:
        X_train, X_test, y_train, y_test, scaler, class_weights
    """
    # Split with stratification
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    
    # Select scaler
    if scaler_type == 'standard':
        scaler = StandardScaler()
    elif scaler_type == 'robust':
        scaler = RobustScaler()  # Less sensitive to outliers
    else:
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
    
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # Compute class weights if requested
    class_weights = None
    if balance_classes:
        classes = np.unique(y_train)
        weights = compute_class_weight('balanced', classes=classes, y=y_train)
        class_weights = dict(zip(classes, weights))
        print(f"\nClass weights: {class_weights}")
    
    return X_train, X_test, y_train, y_test, scaler, class_weights


In [11]:
def train_evaluate(X_train, X_test, y_train, y_test, use_cv=True):
    """
    Train LDA classifier with optional cross-validation.
    """
    clf = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    
    if use_cv:
        # Perform 5-fold cross-validation on training set
        cv_scores = cross_val_score(clf, X_train, y_train, cv=5)
        print(f"Cross-validation scores: {cv_scores}")
        print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    
    # Train on full training set
    clf.fit(X_train, y_train)
    
    # Evaluate on test set
    y_pred = clf.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    
    print(f"\nTest Accuracy: {test_acc:.4f}")
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    
    return clf

In [12]:
if __name__ == "__main__":
    # Configuration
    trial_files = [
        "20251113-Data/Lifting.csv",
        "20251113-Data/Zipping.csv",
        "20251113-Data/Pinching.csv"
    ]
    labels = [0, 1, 2]
    class_names = ['Lifting', 'Zipping', 'Pinching']
    
    # Parameters
    MAX_SENSORS = None  # Use all available sensors (set to number to limit)
    SCALER_TYPE = 'robust'  # 'standard', 'robust', or 'minmax'
    BALANCE_CLASSES = True  # Apply class weights
    CLASSIFIER = 'lda'  # 'lda', 'svm', or 'rf'
    
    print("="*60)
    print("ENHANCED EMG CLASSIFICATION PIPELINE")
    print("="*60)
    print(f"Configuration:")
    print(f"  - Using all available sensors: {MAX_SENSORS is None}")
    print(f"  - Scaler: {SCALER_TYPE}")
    print(f"  - Class balancing: {BALANCE_CLASSES}")
    print(f"  - Classifier: {CLASSIFIER.upper()}")
    
    # Load all trials
    print(f"\n{'='*60}")
    print("STEP 1: Loading and Processing Data")
    print("="*60)
    X, y, trial_info = load_all_trials(trial_files, labels, max_sensors=MAX_SENSORS)
    
    # Print class distribution
    print(f"\nClass distribution:")
    for i, name in enumerate(class_names):
        count = np.sum(y == i)
        pct = 100 * count / len(y)
        print(f"  {name}: {count} samples ({pct:.1f}%)")
    
    # Prepare data
    print(f"\n{'='*60}")
    print("STEP 2: Splitting and Scaling Data")
    print("="*60)
    X_train, X_test, y_train, y_test, scaler, class_weights = prepare_data(
        X, y, scaler_type=SCALER_TYPE, balance_classes=BALANCE_CLASSES
    )
    print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")
    
    # Train and evaluate
    print(f"\n{'='*60}")
    print("STEP 3: Training and Evaluation")
    print("="*60)
    clf, results = train_evaluate(
        X_train, X_test, y_train, y_test,
        classifier_type=CLASSIFIER,
        class_weights=class_weights,
        use_cv=True,
        class_names=class_names
    )
    
    print(f"\n{'='*60}")
    print("PIPELINE COMPLETE")
    print("="*60)
    print(f"\nTo try different configurations, modify the parameters:")
    print(f"  - CLASSIFIER: 'lda', 'svm', or 'rf'")
    print(f"  - SCALER_TYPE: 'standard', 'robust', or 'minmax'")
    print(f"  - BALANCE_CLASSES: True or False")

ENHANCED EMG CLASSIFICATION PIPELINE
Configuration:
  - Using all available sensors: True
  - Scaler: robust
  - Class balancing: True
  - Classifier: LDA

STEP 1: Loading and Processing Data

Processing trial 1/3: 20251113-Data/Lifting.csv


TypeError: arg must be a list, tuple, 1-d array, or Series

In [ ]:
# Main execution (Formulation of embodiment, ML/DL will compare to ground truth, Theory approach is equation based)
if __name__ == "__main__":
    trial_files = [
        "20251113-Data/Lifting.csv",
        "20251113-Data/Zipping.csv",
        "20251113-Data/Pinching.csv"
    ]
    labels = [0, 1, 2]  # 0: Lifting, 1: Zipping, 2: Pinching

    # Test data loading first
    print("="*60)
    print("STEP 1: Testing Data Loading")
    print("="*60)
    all_loaded = all([test_data_loading(f) for f in trial_files])
    
    if not all_loaded:
        print("\n⚠ Some files failed to load. Please check the error messages above.")
    else:
        print("\n" + "="*60)
        print("STEP 2: Processing and Training")
        print("="*60)
        
        print("\nLoading and processing all trials...")
        X, y = load_all_trials(trial_files, labels)
        print(f"✓ Total samples: {X.shape[0]}, Features per sample: {X.shape[1]}")
        print(f"✓ Class distribution: {dict(zip(['Lifting', 'Zipping', 'Pinching'], np.bincount(y)))}")

        print("\nSplitting and scaling data...")
        X_train, X_test, y_train, y_test, scaler = prepare_data(X, y)
        print(f"✓ Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

        print("\n" + "="*60)
        print("STEP 3: Training and Evaluation")
        print("="*60)
        clf = train_evaluate(X_train, X_test, y_train, y_test, use_cv=True)

STEP 1: Testing Data Loading

Testing data loading for: 20251113-Data/Lifting.csv
✗ Error loading data: arg must be a list, tuple, 1-d array, or Series

Testing data loading for: 20251113-Data/Zipping.csv
✗ Error loading data: arg must be a list, tuple, 1-d array, or Series

Testing data loading for: 20251113-Data/Pinching.csv
✗ Error loading data: arg must be a list, tuple, 1-d array, or Series

⚠ Some files failed to load. Please check the error messages above.
